# 08 — Holographic phase transitions, BTZ thermodynamics, and Strominger 1998

**Spacetime Lab — Phase 8 concept + implementation notebook**

Phase 7 verified the simplest non-trivial check of the holographic entanglement entropy correspondence to bit-exact precision: bulk-side Ryu-Takayanagi (geodesic length in AdS$_3$ divided by $4 G_N$) equals boundary-side Calabrese-Cardy ($(c/3) \log(L_A/\epsilon)$) for a single interval at zero temperature. Phase 8 extends this to **non-trivial situations** where the holographic dictionary actually does work:

1. **The BTZ black hole** as the AdS$_3$ analogue of Schwarzschild. Locally identical to AdS$_3$ but with a horizon, a temperature, and an entropy that come from the *global* identification of the angular coordinate.
2. **Strominger 1998** — the simplest microscopic derivation of black-hole entropy. The BTZ Bekenstein-Hawking entropy $\pi r_+ / (2 G_N)$ exactly equals the **sum of two Cardy formula contributions** (one per Virasoro tower) on the boundary 2D CFT. We verify this bit-exactly.
3. **Finite-temperature Calabrese-Cardy** and the bulk dual via BTZ geodesics. Holographic dictionary at $T > 0$, again bit-exact.
4. **The two-interval phase transition** of Headrick 2010. As two intervals are pulled apart, the bulk RT formula switches between *connected* and *disconnected* candidate geodesic configurations. The mutual information is *exactly zero* in the disconnected phase and turns on continuously at the transition, with a non-analytic kink at cross-ratio $x = 1/2$.

**Three things to remember from this notebook**

- BTZ has **zero local curvature beyond AdS$_3$**. Its horizon, temperature, and entropy come from a global quotient, not from new local geometry. It is the simplest known black hole.
- The holographic dictionary at finite temperature is **bit-exact**, just like at zero temperature in Phase 7. The factor of $1/(4 G_N)$ that converts bulk length to boundary entropy is universal.
- The two-interval mutual information has a **sharp non-analyticity** at the holographic phase transition. This non-analyticity is invisible from any local CFT observable — it is a genuinely *holographic* prediction.

**References**

- Bañados, Teitelboim & Zanelli, *The black hole in three-dimensional space-time*, Phys. Rev. Lett. **69** 1849 (1992)
- Strominger, *Black hole entropy from near-horizon microstates*, [arXiv:hep-th/9712251](https://arxiv.org/abs/hep-th/9712251)
- Calabrese & Cardy 2004, [arXiv:hep-th/0405152](https://arxiv.org/abs/hep-th/0405152)
- Headrick, *Entanglement Renyi entropies in holographic theories*, [arXiv:1006.0047](https://arxiv.org/abs/1006.0047)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from spacetime_lab.metrics import BTZ
from spacetime_lab.holography import (
    brown_henneaux_central_charge,
    cardy_formula,
    thermal_calabrese_cardy,
    geodesic_length_btz,
    ryu_takayanagi_btz,
    verify_btz_against_thermal_calabrese_cardy,
    verify_strominger_btz_cardy,
    cross_ratio,
    two_interval_entropy,
    two_interval_mutual_information,
    two_interval_connected_length,
    two_interval_disconnected_length,
    critical_separation_for_phase_transition,
)

plt.rcParams['figure.figsize'] = (8, 5)

## 1. The BTZ black hole

Discovered by Bañados, Teitelboim and Zanelli in 1992, the BTZ black hole is the AdS$_3$ analogue of Schwarzschild. The non-rotating, uncharged version has the metric

$$ds^2 = -\frac{r^2 - r_+^2}{L^2}\,dt^2 + \frac{L^2}{r^2 - r_+^2}\,dr^2 + r^2\,d\varphi^2,$$

with $\varphi \in [0, 2\pi)$ a periodic angular coordinate. The horizon is at $r = r_+$ and the asymptotic boundary is at $r \to \infty$.

**The remarkable fact**: BTZ has *zero local curvature beyond what AdS$_3$ already has*. It is a **quotient** of AdS$_3$ by a discrete subgroup of its isometry group — specifically, the periodic identification of the angular coordinate. The horizon, the temperature, and the entropy are all consequences of this *global* identification, not of any new local geometry.

We verify this by showing that BTZ satisfies the Einstein-constant curvature equation $R_{\mu\nu} = -2/L^2 \cdot g_{\mu\nu}$ pointwise, just like pure AdS$_3$:

In [ ]:
import time

for rp, L in [(1.0, 1.0), (2.0, 1.0), (1.5, 2.5)]:
    bh = BTZ(horizon_radius=rp, ads_radius=L)
    t0 = time.time()
    residual = bh.verify_einstein_constant_curvature()
    elapsed = time.time() - t0
    print(f'BTZ(r_+={rp}, L={L}): max |R_munu - c g_munu| = {residual:.3e}  ({elapsed:.2f}s)')

All zero. BTZ is locally AdS$_3$, as advertised.

## 2. BTZ thermodynamics

The horizon at $r = r_+$ has the standard thermodynamic properties:

| Quantity | Formula |
|---|---|
| Hawking temperature | $T_H = r_+ / (2\pi L^2)$ |
| Mass parameter | $M = r_+^2 / (8 G_N L^2)$ |
| Bekenstein-Hawking entropy | $S_{BH} = \pi r_+ / (2 G_N)$ |

Notice that the entropy scales linearly with $r_+$ (not $r_+^2$ as in higher dimensions) because in 2+1 dimensions the "area" of the horizon is just the *circumference* $2\pi r_+$. The first law $dM = T\,dS$ holds.

In [ ]:
print(f'{"r_+":>5s}  {"T_H":>12s}  {"S_BH":>12s}  {"M":>12s}  {"beta":>12s}')
print('-' * 60)
for rp in [0.5, 1.0, 2.0, 4.0]:
    bh = BTZ(horizon_radius=rp, ads_radius=1.0)
    print(f'{rp:5.2f}  {bh.hawking_temperature():12.6f}  '
          f'{bh.bekenstein_hawking_entropy():12.6f}  {bh.mass_parameter():12.6f}  '
          f'{bh.thermal_beta():12.6f}')

## 3. Strominger 1998 — the simplest microscopic derivation of black-hole entropy

Here is one of the most beautiful results in theoretical physics. **The BTZ Bekenstein-Hawking entropy can be derived from boundary CFT considerations alone**, without ever computing a horizon area. This was Strominger's 1998 observation, which preceded the formal AdS/CFT statement of Maldacena (1997) by months.

The Cardy formula in 2D CFT gives the asymptotic density of high-energy states in each chirality:

$$S_\text{Cardy} = 2\pi\sqrt{\frac{c L_0}{6}} + 2\pi\sqrt{\frac{c \bar L_0}{6}}.$$

For non-rotating BTZ the two chiralities are symmetric: $L_0 = \bar L_0 = M L / 2$, where $M$ is the BTZ mass parameter and $L$ is the AdS radius. With Brown-Henneaux $c = 3 L / (2 G_N)$ and the BTZ formulas, the algebra reduces to:

$$S_\text{Cardy} = 2 \cdot 2\pi\sqrt{\frac{c\,M L / 2}{6}} = 2 \cdot 2\pi \cdot \frac{r_+}{8 G_N} = \frac{\pi r_+}{2 G_N} = S_{BH}.$$

**Bit-exact agreement.** Let's verify it numerically:

In [ ]:
print(f'{"r_+":>5s}  {"L":>5s}  {"S_BH":>14s}  {"S_Cardy":>14s}  {"residual":>12s}')
print('-' * 60)
for rp, L in [(1.0, 1.0), (2.0, 1.0), (1.5, 2.5), (5.0, 3.0), (0.7, 0.5)]:
    s_bh, s_cardy, res = verify_strominger_btz_cardy(rp, L)
    print(f'{rp:5.2f}  {L:5.2f}  {s_bh:14.10f}  {s_cardy:14.10f}  {res:12.2e}')

Bit-exact (or at most one ULP). **The BTZ black hole entropy has a microscopic CFT origin**, which is the proof of concept for the holographic principle that everything in Phases 7-9 builds on.

## 4. Finite-temperature Calabrese-Cardy

The 2D CFT entanglement entropy of an interval at finite temperature $T = 1/\beta$ is

$$S_A^{T>0} = \frac{c}{3} \log\!\left[\frac{\beta}{\pi\epsilon}\sinh\!\left(\frac{\pi L_A}{\beta}\right)\right].$$

This is the Calabrese-Cardy 2004 result. It has two important limits:

- **Low temperature** ($\beta \gg L_A$): $\sinh(\pi L_A/\beta) \to \pi L_A/\beta$, and the formula collapses to $(c/3) \log(L_A/\epsilon)$ — Phase 7's zero-temperature result.
- **High temperature** ($\beta \ll L_A$): $\sinh(\pi L_A/\beta) \to (1/2) e^{\pi L_A/\beta}$, and the formula becomes $(c/3)[\log(\beta/(2\pi\epsilon)) + \pi L_A/\beta]$. The second term is the standard 1+1 CFT thermal entropy density $\pi c L_A / (3\beta)$.

**The bulk dual is the BTZ geodesic length**, with the same closed form (the AdS$_3$ semicircle is replaced by a curve in BTZ that goes "around" the horizon). The bit-exact agreement of bulk and boundary at finite temperature is the Phase 8 finite-T gate test:

In [ ]:
print(f'{"L_A":>6s}  {"r_+":>5s}  {"L":>5s}  {"eps":>10s}  {"S_RT":>14s}  {"S_CC":>14s}  {"residual":>12s}')
print('-' * 80)
for L_A, rp, L_ads, eps in [
    (2.0, 1.0, 1.0, 0.01),
    (5.0, 2.0, 1.0, 0.001),
    (10.0, 3.0, 2.0, 1e-4),
    (1.0, 1.0, 0.5, 0.001),
    (100.0, 1.0, 1.0, 1e-6),  # high-T regime
]:
    rt, cc, res = verify_btz_against_thermal_calabrese_cardy(L_A, rp, L_ads, eps)
    print(f'{L_A:6.2f}  {rp:5.2f}  {L_ads:5.2f}  {eps:10.2e}  {rt:14.10f}  {cc:14.10f}  {res:12.2e}')

Every residual at exactly 0.00e+00, including the high-temperature case where the naive $\sinh$ would overflow (we use a numerically stable $\log\sinh$ helper).

## 5. The interpolation: low-T to high-T

Let's plot the finite-temperature entanglement entropy as a function of interval length $L_A$ at three different temperatures and compare against the two limiting formulas. The crossover from logarithmic (low T) to linear (high T) is where the *thermal* entropy starts to dominate over the entanglement entropy.

In [ ]:
c = 1.5  # central charge
epsilon = 0.001
L_A_values = np.logspace(-1, 2, 200)

fig, ax = plt.subplots(figsize=(9, 5))
for beta, color in [(20.0, '#0050a0'), (5.0, '#1a9a4a'), (0.5, '#c64a0b')]:
    S = [thermal_calabrese_cardy(L_A, central_charge=c, beta=beta, epsilon=epsilon)
         for L_A in L_A_values]
    ax.plot(L_A_values, S, color=color, lw=2.0, label=f'$\\beta = {beta}$')

# Zero-temperature limit
from spacetime_lab.holography import calabrese_cardy_2d
S_zeroT = [calabrese_cardy_2d(L_A, central_charge=c, epsilon=epsilon) for L_A in L_A_values]
ax.plot(L_A_values, S_zeroT, color='gray', lw=1.0, ls='--', label='zero-T limit ($(c/3) \\log(L_A/\\epsilon)$)')

ax.set_xscale('log')
ax.set_xlabel(r'interval length $L_A$ (log scale)')
ax.set_ylabel(r'$S_A^{T>0}$ (nats)')
ax.set_title(f'Finite-temperature entanglement entropy ($c = {c}$, $\\epsilon = {epsilon}$)')
ax.legend(loc='upper left')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

Read this plot like this:

- At small $L_A$ (left side), all curves agree with the zero-temperature dashed gray curve. **Below the thermal scale $\beta$, the temperature is invisible to the entanglement entropy.**
- At large $L_A$ (right side), the curves split. The hot ($\beta = 0.5$) curve grows linearly with slope $\pi c / (3\beta) \approx 3.14$, while the cold ($\beta = 20$) curve still grows logarithmically. **Above the thermal scale, the entanglement entropy is dominated by ordinary thermal entropy.**
- The crossover happens at $L_A \sim \beta$, exactly where you'd expect on dimensional grounds.

## 6. The two-interval phase transition (Headrick 2010)

Now we go from a single interval to **two**. Take two disjoint intervals $A_1 = [a, b]$ and $A_2 = [c, d]$ with $a < b < c < d$. The bulk has *two* candidate minimal-surface configurations:

**Disconnected**: each interval has its own bulk geodesic (semicircle in $\mathbb{H}^2$).
$$\mathcal{L}^D = 2L\log\frac{b-a}{\epsilon} + 2L\log\frac{d-c}{\epsilon}.$$

**Connected**: the four endpoints are linked by *two new* geodesics — one from $a$ to $d$ (the "outer" geodesic) and one from $b$ to $c$ (the "inner" geodesic).
$$\mathcal{L}^C = 2L\log\frac{d-a}{\epsilon} + 2L\log\frac{c-b}{\epsilon}.$$

**The Ryu-Takayanagi prescription picks the minimum.** As the gap shrinks, $\mathcal{L}^C$ becomes smaller than $\mathcal{L}^D$ and the connected phase takes over. The transition is governed by the **cross-ratio**

$$x = \frac{(b-a)(d-c)}{(c-a)(d-b)},$$

with the connected phase winning when $x > 1/2$ and disconnected when $x < 1/2$. The critical cross-ratio is $x = 1/2$ exactly — this is the **Headrick 2010** result.

In [ ]:
# Critical separation for two unit intervals
L0 = 1.0
d_crit = critical_separation_for_phase_transition(L0, L0)
print(f'Critical gap for two intervals of length {L0}:  {d_crit:.10f}')
print(f'Closed form:  sqrt(2) - 1 = {math.sqrt(2.0) - 1.0:.10f}')
print()
x_at_critical = cross_ratio(0.0, L0, L0 + d_crit, 2 * L0 + d_crit)
print(f'Cross-ratio at critical gap: {x_at_critical:.10f}  (should be 0.5)')

In [ ]:
# Tabulate the phase transition: equal intervals of length 1, varying gap
L0 = 1.0
print(f'{"gap d":>8s}  {"x":>10s}  {"L^D":>10s}  {"L^C":>10s}  {"phase":>13s}  {"I(A:B)":>10s}')
print('-' * 70)
for d in [0.05, 0.1, 0.2, 0.3, d_crit, 0.5, 0.7, 1.0, 2.0, 5.0]:
    a, b, c, dd = 0.0, L0, L0 + d, 2 * L0 + d
    L_D = two_interval_disconnected_length(a, b, c, dd, ads_radius=1.0, epsilon=0.001)
    L_C = two_interval_connected_length(a, b, c, dd, ads_radius=1.0, epsilon=0.001)
    result = two_interval_entropy(a, b, c, dd, ads_radius=1.0, epsilon=0.001)
    I = two_interval_mutual_information(a, b, c, dd, ads_radius=1.0, epsilon=0.001)
    x = result['cross_ratio']
    print(f'{d:8.4f}  {x:10.6f}  {L_D:10.6f}  {L_C:10.6f}  {result["phase"]:>13s}  {I:10.6f}')

The phase transition is sharp: the minimum length switches from $\mathcal{L}^C$ (connected) to $\mathcal{L}^D$ (disconnected) exactly at the critical gap $d = \sqrt{2} - 1 \approx 0.4142$, where the cross-ratio is exactly $1/2$.

## 7. Mutual information vs separation — the headline plot

The mutual information $I(A_1 : A_2) = S(A_1) + S(A_2) - S(A_1 \cup A_2)$ is the most striking observable to plot. **It is exactly zero in the disconnected phase** (because $S(A_1 \cup A_2) = S(A_1) + S(A_2)$ when each interval has its own RT geodesic), and **positive in the connected phase**. The transition from zero to positive is *not smooth* — it has a kink at the critical separation.

In [ ]:
L0 = 1.0
d_crit = critical_separation_for_phase_transition(L0, L0)

gaps = np.linspace(0.01, 2.0, 400)
I_values = np.array([
    two_interval_mutual_information(
        0.0, L0, L0 + d, 2 * L0 + d, ads_radius=1.0, epsilon=0.001
    )
    for d in gaps
])

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(gaps, I_values, color='#0050a0', lw=2.0)
ax.axvline(d_crit, color='#a40000', lw=1.5, ls='--',
           label=f'critical gap $d_{{\\rm crit}} = \\sqrt{{2}} - 1 \\approx {d_crit:.4f}$')
ax.axhline(0, color='k', lw=0.5)
ax.set_xlabel(r'gap between intervals $d$')
ax.set_ylabel(r'holographic mutual information $I(A_1 : A_2)$')
ax.set_title(r'Two-interval phase transition (equal intervals of length $L_0 = 1$)')
ax.set_xlim(0, 2.0)
ax.set_ylim(-0.1, 3.0)
ax.legend(loc='upper right')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Notice three features:

1. **For $d > d_\text{crit}$, the mutual information is exactly zero.** This is not a small numerical leakage — it is mathematically zero because the disconnected configuration trivially gives $S(A_1 \cup A_2) = S(A_1) + S(A_2)$.
2. **At $d = d_\text{crit}$ there is a sharp kink.** The mutual information turns on with finite slope, not smoothly. This is the holographic non-analyticity.
3. **As $d \to 0$ the mutual information diverges**, reflecting the cutoff-dependent UV divergence of joining two adjacent intervals.

**The non-analyticity at $d = d_\text{crit}$ is invisible from any local CFT observable**. It is a genuinely *holographic* prediction: the existence of the bulk minimal-surface phase transition manifests itself on the boundary as a non-smooth feature in the mutual information. Headrick (2010) checked this against direct CFT calculations of the Renyi entropies and found agreement.

## 8. Phase 8 gate checks

Before we move on the following must hold.

In [ ]:
# (1) BTZ is locally AdS_3
for rp, L in [(1.0, 1.0), (2.0, 1.5), (0.5, 0.7)]:
    bh = BTZ(horizon_radius=rp, ads_radius=L)
    assert bh.verify_einstein_constant_curvature() < 1e-10

# (2) BTZ thermodynamics: T_H = r_+ / (2 pi L^2), S_BH = pi r_+ / (2 G_N)
bh = BTZ(horizon_radius=2.0, ads_radius=1.0)
assert math.isclose(bh.hawking_temperature(), 2.0 / (2 * math.pi))
assert math.isclose(bh.bekenstein_hawking_entropy(), math.pi)

# (3) Strominger 1998: BTZ S_BH = sum of two Cardy contributions
for rp, L in [(1.0, 1.0), (2.0, 1.0), (1.5, 2.5), (5.0, 3.0)]:
    _, _, residual = verify_strominger_btz_cardy(rp, L)
    assert residual < 1e-12

# (4) Finite-T RT in BTZ equals thermal Calabrese-Cardy bit-exactly
for L_A, rp, L_ads, eps in [
    (2.0, 1.0, 1.0, 0.01),
    (10.0, 3.0, 2.0, 1e-4),
    (100.0, 1.0, 1.0, 1e-6),
]:
    _, _, residual = verify_btz_against_thermal_calabrese_cardy(L_A, rp, L_ads, eps)
    assert residual < 1e-12

# (5) Low-T limit reduces to zero-T Calabrese-Cardy
S_zeroT = calabrese_cardy_2d(2.0, central_charge=1.5, epsilon=0.01)
S_lowT = thermal_calabrese_cardy(2.0, central_charge=1.5, beta=1e6, epsilon=0.01)
assert math.isclose(S_lowT, S_zeroT, abs_tol=1e-9)

# (6) Critical gap for two unit intervals is sqrt(2) - 1
d_crit = critical_separation_for_phase_transition(1.0, 1.0)
assert math.isclose(d_crit, math.sqrt(2.0) - 1.0, abs_tol=1e-12)

# (7) Cross-ratio at the critical gap is exactly 1/2
x_crit = cross_ratio(0.0, 1.0, 1.0 + d_crit, 2.0 + d_crit)
assert math.isclose(x_crit, 0.5, abs_tol=1e-12)

# (8) Mutual information is exactly zero in the disconnected phase
I = two_interval_mutual_information(0.0, 1.0, 5.0, 6.0, ads_radius=1.0, epsilon=0.001)
assert I == 0.0

# (9) Mutual information is positive in the connected phase
I = two_interval_mutual_information(0.0, 1.0, 1.05, 2.05, ads_radius=1.0, epsilon=0.001)
assert I > 0

# (10) Mutual information is cutoff-independent
I1 = two_interval_mutual_information(0.0, 1.0, 1.1, 2.1, ads_radius=1.0, epsilon=0.001)
I2 = two_interval_mutual_information(0.0, 1.0, 1.1, 2.1, ads_radius=1.0, epsilon=1e-8)
assert math.isclose(I1, I2, abs_tol=1e-12)

print('ALL PHASE 8 GATE CHECKS PASSED')

---

## What's next — Phase 9 teaser

Phase 9 enters the modern frontier: the **island formula** (Penington 2019; Almheiri, Engelhardt, Marolf, Maxfield 2019). This is the resolution of the Hawking information paradox via *quantum corrections* to the Ryu-Takayanagi formula.

For an evaporating black hole, the von Neumann entropy of the radiation should follow the **Page curve**: rising at first, then falling back to zero once the black hole has fully evaporated. Hawking's 1976 calculation gives a monotonically rising curve — apparently violating unitarity. The island formula resolves this by adding a *quantum extremal surface* term to RT:

$$S_\text{rad} = \min_X \, \text{ext}_X\!\left[\frac{\text{Area}(\partial X)}{4 G_N} + S_\text{semi-classical}(\text{Rad} \cup X)\right].$$

When the black hole is young, the trivial $X = \emptyset$ saddle wins and you get the Hawking rising curve. When the black hole is old (after the Page time), a non-trivial 'island' $X$ inside the BH wins and the entropy starts to *decrease*. The transition between the two saddles is **a holographic phase transition exactly like the two-interval one we just plotted in this notebook** — same kind of mathematical structure, but in a completely different physical context.

Phase 9 will implement the simplest analytic island formula example (a 2D dilaton-gravity model on AdS$_2$, where everything is computable) and verify the Page curve appears correctly.

## Honest scope notes for Phase 8

- **Only the non-rotating, uncharged BTZ** is implemented. Rotating BTZ ($J \neq 0$) has inner and outer horizons and a richer thermodynamic structure (extremal limit, ergoregion). Deferred.
- **Only the equatorial-observer two-interval entropy** (no time-dependent / HRT generalisation). Deferred.
- **No numerical bulk minimal-surface finder for higher-dimensional AdS**. The closed-form AdS$_3$/BTZ cases give us bit-exact verification of every Phase 8 deliverable, which is plenty. A general numerical surface finder lands in a v0.8.1 patch when there is a higher-dimensional case that genuinely needs it.
- **No Lewkowycz-Maldacena 2013 'derivation' of RT**. That is a beautiful but entirely analytical argument that does not become more illuminating by being coded up. Mention in the notebook prose, do not implement.